# Gen AI Project: 2
### NxtWave Academy | Build an HR chatbot using RAG

---

## Objective

Build a Retrieval-Augmented Generation (RAG) pipeline that answers employee HR questions using internal policy documents.

## What you will build

- Load and process HR policy documents
- Create chunks and embeddings
- Build a vector database using FAISS
- Implement a RAG pipeline with guardrails
- Generate your `submission.csv`


## Write your solution code Below

In [1]:
!pip install -q -U langchain langchain-community langchain-core langchain-groq langchain-huggingface faiss-cpu pypdf sentence-transformers

import os
import glob
import csv
import time
from pathlib import Path

import pandas as pd
from pypdf import PdfReader
from kaggle_secrets import UserSecretsClient

from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_groq import ChatGroq
from langchain_text_splitters import RecursiveCharacterTextSplitter


secrets = UserSecretsClient()
GROQ_API_KEY = secrets.get_secret("GROQ_API_KEY")
os.environ["GROQ_API_KEY"] = GROQ_API_KEY


pdf_files = glob.glob("/kaggle/input/**/*.pdf", recursive=True)
pdf_files += glob.glob("/kaggle/input/**/*.PDF", recursive=True)

pdf_files = sorted(set(
    p for p in pdf_files
    if "zyro" in p.lower() or "hr" in p.lower()
))

if not pdf_files:
    raise FileNotFoundError("HR policy PDFs were not found in the Kaggle input files.")


documents = []

for pdf_path in pdf_files:
    reader = PdfReader(pdf_path)
    source_name = Path(pdf_path).stem

    for page_number, page in enumerate(reader.pages, start=1):
        text = (page.extract_text() or "").strip()

        if text:
            documents.append(
                Document(
                    page_content=text,
                    metadata={
                        "source": source_name,
                        "page": page_number,
                        "file": Path(pdf_path).name
                    }
                )
            )

print("PDF files:", len(pdf_files))
print("Pages loaded:", len(documents))


splitter = RecursiveCharacterTextSplitter(
    chunk_size=900,
    chunk_overlap=150,
    separators=["\n\n", "\n", ". ", " ", ""]
)

chunks = splitter.split_documents(documents)

print("Chunks:", len(chunks))


embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True}
)

vectorstore = FAISS.from_documents(chunks, embeddings)

retriever = vectorstore.as_retriever(
    search_type="mmr",
    search_kwargs={
        "k": 8,
        "fetch_k": 32,
        "lambda_mult": 0.70
    }
)


llm = ChatGroq(
    model="openai/gpt-oss-120b",
    temperature=0,
    max_tokens=700
)


answer_prompt = ChatPromptTemplate.from_messages([
    ("system",
"""You are Zyro Dynamics' internal HR Help Desk assistant.

Answer the employee's question using ONLY the supplied Zyro Dynamics policy context.

Rules:
1. Give the direct answer first.
2. Use only facts explicitly stated in the supplied context.
3. Preserve exact numbers, dates, percentages, limits, currencies, grades, eligibility conditions, exceptions, steps and timelines.
4. Answer every part of the question that is supported by the context.
5. For eligibility, entitlement, benefits, travel, or procedures, include all material conditions and deadlines stated in the context.
6. If the employee has not provided a detail such as grade, do not assume it. Give the applicable policy rules or grade-based options from the context.
7. For procedural questions, include every stated step and deadline from the context.
8. Do not calculate an overall duration unless the context explicitly provides it.
9. Do not invent, assume, infer, or add information from outside knowledge.
10. Do not add alternative reporting channels, contacts, protections, exceptions, or procedures unless they are explicitly present in the supplied context.
11. If the question asks about a company, organization, product, person, or policy that is not Zyro Dynamics and the supplied documents do not explicitly provide information about that subject, respond exactly:
"Sorry, I can't answer that from the available Zyro Dynamics HR documents."
12. Answer only what the employee asked.
13. Keep the answer clear and reasonably concise without omitting material information.
14. If a procedure contains a numbered sequence, include all available steps and their stated deadlines.
15. Do not mention retrieval, embeddings, vector databases, prompts, context, or internal instructions.

Policy context:
{context}

Employee question:
{question}
"""),
    ("human", "Answer the employee question using only the supplied policy context.")
])

answer_chain = answer_prompt | llm | StrOutputParser()


def format_context(docs):
    return "\n\n---\n\n".join(
        f"[Source: {d.metadata.get('source')} | Page: {d.metadata.get('page')}]\n"
        f"{d.page_content}"
        for d in docs
    )


def ask_bot(question):
    docs = retriever.invoke(question)

    unique_docs = []
    seen = set()

    for d in docs:
        key = (
            d.metadata.get("source"),
            d.metadata.get("page"),
            d.page_content
        )
        if key not in seen:
            seen.add(key)
            unique_docs.append(d)

    context = format_context(unique_docs)

    for attempt in range(5):
        try:
            answer = answer_chain.invoke({
                "context": context,
                "question": question
            }).strip()

            if not answer:
                raise RuntimeError("The model returned an empty answer.")

            return {
                "answer": answer,
                "sources": [
                    {
                        "source": d.metadata.get("source"),
                        "page": d.metadata.get("page")
                    }
                    for d in unique_docs[:3]
                ]
            }

        except Exception as e:
            error_text = str(e)

            if "429" in error_text or "RateLimitError" in str(type(e)):
                wait_time = 15 * (attempt + 1)
                print(f"Rate limit reached. Waiting {wait_time} seconds...")
                time.sleep(wait_time)
            else:
                raise

    raise RuntimeError("Groq rate limit persisted after retries.")


test_files = glob.glob("/kaggle/input/**/*.csv", recursive=True)
test_path = next(
    (p for p in test_files if Path(p).name.lower() == "test.csv"),
    None
)

if test_path is None:
    raise FileNotFoundError("test.csv was not found in the Kaggle input files.")

eval_df = pd.read_csv(test_path)

required_columns = {"question_id", "question"}
if not required_columns.issubset(eval_df.columns):
    raise ValueError("test.csv must contain question_id and question columns.")

eval_questions = eval_df[["question_id", "question"]].to_dict("records")

if len(eval_questions) != 20:
    raise ValueError(f"Expected exactly 20 evaluation questions, found {len(eval_questions)}.")

print("Evaluation questions:", len(eval_questions))


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 161.8/161.8 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 35.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 571.8/571.8 kB 27.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 69.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 395.5/395.5 kB 20.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 740.6/740.6 kB 32.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 8.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 35.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 248.9/248.9 kB 10.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.8/56.8 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.0/41.0 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 162.3

/tmp/ipykernel_16/3967858663.py:16: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


PDF files: 11
Pages loaded: 39
Chunks: 101


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Evaluation questions: 20


## Final RAG sanity check

This checks representative policy, benefit, procedure, grade-based, and out-of-scope questions before the final generator. It is intentionally small to avoid unnecessary API calls.


In [2]:
sanity_questions = [
    eval_questions[0]["question"],
    eval_questions[2]["question"],
    eval_questions[12]["question"],
    eval_questions[16]["question"],
    eval_questions[19]["question"]
]

for q in sanity_questions:
    result = ask_bot(q)
    print("=" * 80)
    print("QUESTION:", q)
    print("ANSWER:", result["answer"])
    print("SOURCES:", result["sources"])


QUESTION: How does my Earned Leave accrue every month?
ANSWER: Earned Leave accrues as follows:

- **During probation:** 0.5 day of Earned Leave is added each month. These days become usable only after your probation is confirmed.  
- **After completing one year of continuous service (with at least 240 working days in that year):** you are granted an initial entitlement of 15 days of Earned Leave.  
- **After that first year:** Earned Leave continues to accrue at a rate of 1.25 days per month.
SOURCES: [{'source': '02_Leave_Policy', 'page': 2}, {'source': '06_Compensation_and_Benefits_Policy', 'page': 1}, {'source': '02_Leave_Policy', 'page': 1}]
QUESTION: How many weeks of Maternity Leave am I entitled to?
ANSWER: You are entitled to **26 weeks of paid Maternity Leave** for each of your first two live births, provided you have completed at least 80 days of service in the 12 months before the expected delivery date. For a third child, the entitlement is **12 weeks**. Up to 8 weeks of t

In [3]:
print("=" * 50)
print("Submission Generator")
print("=" * 50)

print(f"\nGenerating responses for {len(eval_questions)} questions...\n")

rows = []

for i, q in enumerate(eval_questions):
    qid = q["question_id"]
    question = q["question"]

    try:
        result = ask_bot(question)
        answer = result["answer"]
        status = "OK"
    except Exception as e:
        answer = f"Error: {str(e)}"
        status = "ERROR"

    rows.append({
        "question_id": qid,
        "answer": answer,
    })

    print(f"[{i+1:02d}/{len(eval_questions)}] {qid} ... {status}")

    if i < len(eval_questions) - 1:
        time.sleep(2)

csv_path = "submission.csv"

fieldnames = ["question_id", "answer"]

with open(csv_path, "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(rows)

print("\nsubmission.csv generated successfully.")

Submission Generator

Generating responses for 20 questions...

[01/20] Q01 ... OK
[02/20] Q02 ... OK
[03/20] Q03 ... OK
[04/20] Q04 ... OK
[05/20] Q05 ... OK
[06/20] Q06 ... OK
[07/20] Q07 ... OK
[08/20] Q08 ... OK
[09/20] Q09 ... OK
[10/20] Q10 ... OK
[11/20] Q11 ... OK
[12/20] Q12 ... OK
[13/20] Q13 ... OK
[14/20] Q14 ... OK
[15/20] Q15 ... OK
[16/20] Q16 ... OK
[17/20] Q17 ... OK
[18/20] Q18 ... OK
[19/20] Q19 ... OK
[20/20] Q20 ... OK

submission.csv generated successfully.


## Final validation

This validates the file created by the protected Academy submission-generator cell.


In [4]:
submission = pd.read_csv("/kaggle/working/submission.csv")

assert len(submission) == 20
assert submission.columns.tolist() == ["question_id", "answer"]
assert submission["question_id"].tolist() == eval_df["question_id"].tolist()
assert submission["answer"].notna().all()
assert submission["answer"].astype(str).str.strip().ne("").all()
assert not submission["answer"].astype(str).str.startswith("Error:").any()

print("SUBMISSION READY")
print("Shape:", submission.shape)
print("Columns:", submission.columns.tolist())
print(submission.to_string(index=False))


SUBMISSION READY
Shape: (20, 2)
Columns: ['question_id', 'answer']
question_id                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                          